In [15]:
from matplotlib import pyplot as plt
import seaborn as sns
import json
import tqdm

import numpy as np
import pandas as pd

from bleurt import score


In [2]:
datasets: dict = {}  # Dictionary to hold per-model DataFrames

baseline_file =     ('evaluation_translations/smolsent_test_translations_original_madlad.json')  # Baseline file path
token_swap_file =   ('evaluation_translations/smolsent_test_tokenswap_translations_original_madlad.json')  # Token replacement file path
finetuned_file =    ('evaluation_translations/smolsent_test_translations_finetuned_madlad.json')  # Fine-tuned model file path
ts_finetuned_file = ('evaluation_translations/smolsent_test_tokenswap_translations_finetuned_madlad.json')  # Fine-tuned token replacement file path
file_paths = {
    'Baseline': baseline_file,
    'Token Swap': token_swap_file,
    'Fine-tuned': finetuned_file,
    'TS Fine-tuned': ts_finetuned_file
}

# Read each file and keep only src, trg and the model's translation as a column named by the label
translation_key = 'madlad_translation'
for label, file_path in file_paths.items():
    with open(file_path, 'r') as f:
        data = json.load(f)
        df = pd.DataFrame(data)
        # find translation column (fallback to any column containing 'trans')
        trans_col = translation_key if translation_key in df.columns else next((c for c in df.columns if 'trans' in c.lower()), None)
        if trans_col is None:
            raise KeyError(f"No translation column found in {file_path}")
        df = df[['src', 'trg', trans_col]].rename(columns={trans_col: label})
        trg_label = 'trg' if 'swap' not in file_path.lower() else 'trg_swapped'
        df: pd.DataFrame = df.rename(columns={'trg': trg_label})
        print(df.columns)
        datasets[label] = df

# Merge all model DataFrames on src and trg so each row has translations from every model
from functools import reduce
merged_swapped_df = pd.merge(datasets['Token Swap'], datasets['TS Fine-tuned'], on=['src', 'trg_swapped'], how='outer')
merged_org_df = pd.merge(datasets['Baseline'], datasets['Fine-tuned'], on=['src', 'trg'], how='outer')
merged_df = pd.merge(merged_org_df, merged_swapped_df, on=['src'], how='outer')
# Remove duplicate source sentences (if any) and reset index
merged_df = merged_df.drop_duplicates(subset=['src']).reset_index(drop=True)
# Display the first few rows of the merged (wide) DataFrame
merged_df.head(100)

Index(['src', 'trg', 'Baseline'], dtype='object')
Index(['src', 'trg_swapped', 'Token Swap'], dtype='object')
Index(['src', 'trg', 'Fine-tuned'], dtype='object')
Index(['src', 'trg_swapped', 'TS Fine-tuned'], dtype='object')


,src,trg,Baseline,Fine-tuned,trg_swapped,Token Swap,TS Fine-tuned
0,A warehouse worker in a retail store is holdin...,間零售店嘅倉務員拎住部手機，同埋準備打包個包裹寄去海外。,The storekeeper in the retail store takes the ...,The warehouse clerk at a retail store is holdi...,間零售店的倉務員拎居住部手機，和埋準備打包那个包裹寄去海外。,The warehouseman at the retail store took the ...,The warehouse clerk at a retail store is carry...
1,Accessories lend visual appeal through pattern...,配飾通過圖案同質感令視覺外觀更吸引。,The decorations are made more attractive by th...,Decoration makes the visual appearance more ap...,配飾通过图案和質感令视觉外觀更吸引。,Decoration makes the visual appearance more at...,Decoration adds visual appeal to a space throu...
2,"Adaptability - flexible, agile mindset, optimi...",適應能力 - 彈性同靈活嘅思維，積極樂觀嘅態度；,Adaptability - flexibility and agility of thin...,"Adaptation - resilience and agile thinking, po...",适应能力 - 弹性和靈活的思維，積極乐观的态度；,Adaptability - resilience and flexibility of m...,"Adaptation - resilient and flexible thinking, ..."
3,"As a reminder, this company launched a 34-inch...",提醒返，呢間公司舊年十月推出咗一款34吋曲面遊戲顯示器。,"In October 2011, the company announced a 34-in...",Recall that the company launched a 34-inch cur...,提醒返，呢間公司舊年十月推出咗一款34吋曲脸游戏顯示器。,"In October 2011, the company launched a 34-inc...",Recall that the company launched a 34-inch cur...
4,"As a result, the teeth eventually develop a du...",結果，啲牙最終會變到暗啞無光。,"As a result, the teeth eventually become dull ...","As a result, the teeth eventually become dull ...",结果，啲牙最終会變到暗啞無亮。,"As a result, the teeth eventually become dull ...","As a result, the teeth eventually become dull ..."
...,...,...,...,...,...,...,...
82,"We might be inspired by books, concepts, or hi...",我哋可能會受到書本、概念或者歷史人物嘅啟發。,"We may be inspired by books, concepts, or hist...","We may be inspired by books, concepts, or hist...",我哋可能会受到书本、概念或者历史人物的啟發。,"We may be inspired by books, concepts or histo...","We may be inspired by books, concepts, or hist..."
83,We will most likely see an extension of the cu...,我哋好大機會會延長現時嘅封鎖措施，同時我哋嘅政府正加快緊疫苗接種嘅工作。,"We are likely to extend the current lockdowns,...",We are likely to extend our current lockdown m...,我哋好大机会会延长現時的封锁措施，和時我哋的政府正加快紧疫苗接種的工作。,We have a good chance of extending the current...,We are likely to extend our current lockdown m...
84,With the advent of sophisticated surveillance ...,隨住先進嘅監控同創新科技嘅出現，一定要謹慎行事。,The emergence of advanced monitoring and innov...,With the advent of advanced monitoring and inn...,隨居住先进的監控和创新科技的出现，一定要謹慎走事。,With the emergence of advanced monitoring and ...,With the advent of advanced monitoring and inn...
85,Withdrawal effects are small and more annoying...,戒斷反應唔大，同埋係煩多過危險。,The withdrawal is not very effective and is mo...,"It's not a good response, and it's more of a n...",戒斷反应不大，和埋是煩多過危险。,"The withdrawal reaction is not great, and the ...","The withdrawal was not very effective, and it ..."


In [3]:
scorer = score.BleurtScorer("BLEURT-20")
import tensorflow as tf

INFO:tensorflow:Reading checkpoint BLEURT-20.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: BLEURT-20/sent_piece.model.
INFO:tensorflow:SentencePiece tokenizer created.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.


I0000 00:00:1765910791.386500  147707 gpu_device.cc:2421] Ignoring visible gpu device (device: 0, name: AMD Radeon Graphics, pci bus id: 0000:04:00.0) with AMDGPU version : gfx1152. The supported AMDGPU versions are gfx900, gfx906, gfx908, gfx90a, gfx942, gfx950, gfx1030, gfx1100, gfx1101, gfx1103, gfx1150, gfx1151, gfx1200, gfx1201.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [5]:
print(tf.version.VERSION)
print(tf.config.list_physical_devices(None))

2.21.0-dev0+selfbuilt
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [6]:
for col in tqdm.tqdm(['Baseline', 'Fine-tuned', 'Token Swap', 'TS Fine-tuned']):
    print(f"Scoring column: {col}")
    references = (merged_df['src'].tolist())
    candidates = merged_df[col].tolist()
    print(references[0])
    scores = scorer.score(references=references, candidates=candidates)
    merged_df[f'{col}_BLEURT'] = scores

  0%|          | 0/4 [00:00<?, ?it/s]

Scoring column: Baseline
A warehouse worker in a retail store is holding a cell phone and preparing to package up a parcel to ship overseas.


 25%|██▌       | 1/4 [03:24<10:14, 204.93s/it]

Scoring column: Fine-tuned
A warehouse worker in a retail store is holding a cell phone and preparing to package up a parcel to ship overseas.


 50%|█████     | 2/4 [06:50<06:50, 205.28s/it]

Scoring column: Token Swap
A warehouse worker in a retail store is holding a cell phone and preparing to package up a parcel to ship overseas.


 75%|███████▌  | 3/4 [10:14<03:24, 204.60s/it]

Scoring column: TS Fine-tuned
A warehouse worker in a retail store is holding a cell phone and preparing to package up a parcel to ship overseas.


100%|██████████| 4/4 [13:37<00:00, 204.33s/it]


In [8]:
#save to csv
merged_df.to_csv('evaluation_translations/combined_smolsent_bleurt_scores_madlad.csv', index=False)

In [22]:
print(f"Baseline:\t\t\t {np.mean(merged_df['Baseline_BLEURT'])}")
print(f"Token swap:\t\t {np.mean(merged_df['Token Swap_BLEURT'])}")
print(f"Fine-tuned:\t\t {np.mean(merged_df['Fine-tuned_BLEURT'])}")
print(f"TS Fine-tuned:\t {np.mean(merged_df['TS Fine-tuned_BLEURT'])}")

Baseline:			 0.6806566989284822
Token swap:		 0.6704189777374268
Fine-tuned:		 0.7254119131756925
TS Fine-tuned:	 0.7144110394620348


In [24]:
with open("unlabelled/character_classification.json", "r") as f:
    character_classfication = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'unlabelled/character_classification.json'